# Phase 34 — v1.4 Latency-Only Protocol-Compliant Re-Execution
## Reviewer Notebook

**Classification:** `READ_ONLY_PHASE34_LATENCY_EXECUTION_V1_4_REVIEWER_NOTEBOOK`
**Revision:** `ra_medfilm_barr_phase34_efficiency_profiling_v1_4`

Six canonical batch-1 arms profiled with **1000 per-forward CUDA-event observations each**
(6000 raw observations). Throughput, memory, static-resource and analytical MAC evidence from
v1.3 is **reused valid, not regenerated**.


## 0. Setup

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(r"D:\Softfors\jup")
AUD = ROOT / "reports" / "audits"
TAB = ROOT / "reports" / "tables"


def jload(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))


def tshow(name, n=None):
    df = pd.read_csv(TAB / name)
    print(name, "| shape:", df.shape)
    if n is not None:
        df = df.head(n)
    with pd.option_context("display.max_columns", None, "display.width", 260):
        print(df.to_string(index=False))
    return df


print("Phase34 v1.4 latency-only protocol-compliant re-execution review")
print("READ-ONLY. No checkpoint load, no model construction, no forward, no timing rerun,")
print("no throughput/memory rerun, no profiler, no protected data, no TEST, no training.")


Phase34 v1.4 latency-only protocol-compliant re-execution review
READ-ONLY. No checkpoint load, no model construction, no forward, no timing rerun,
no throughput/memory rerun, no profiler, no protected data, no TEST, no training.


## 1. Source preflight, environment and provenance

In [2]:
pf = jload(AUD / "phase34_latency_execution_source_preflight_v1_4.json")
print("preflight pass:", pf["all_pass"], "| sources:", pf["all_sources_match"],
      "| preserved v1.3:", pf["preserved_valid_v1_3_all_match"],
      "| checkpoints:", pf["all_checkpoints_match"])
env = jload(AUD / "phase34_latency_execution_environment_v1_4.json")
print("environment gate:", env["environment_gate_pass"])
for k, v in env["environment_checks"].items():
    print(f"  {k:28s} = {v}")
pv = jload(AUD / "phase34_latency_execution_provenance_v1_4.json")
print()
print("six-arm order:", pv["six_arm_order"])
print("physical worker count:", pv["physical_worker_count"],
      "| alias physical worker count:", pv["alias_physical_worker_count"])
print("CUDA-event granularity:", pv["cuda_event_granularity"])


preflight pass: True | sources: True | preserved v1.3: True | checkpoints: True
environment gate: True
  gpu_model                    = True
  gpu_compute_capability       = True
  gpu_total_vram_bytes         = True
  python_version               = True
  torch_version                = True
  cuda_runtime                 = True
  cuda_driver_version          = True
  cudnn_version                = True

six-arm order: ['GLOBAL_ONLY_C1', 'R_PSI_COMPONENT', 'B_PHI_COMPONENT', 'S_OMEGA_COMPONENT', 'SKIP_CONTROLLED_PATH', 'EXECUTE_CONTROLLED_PATH']
physical worker count: 6 | alias physical worker count: 0
CUDA-event granularity: ONE_CUDA_EVENT_PAIR_PER_INDIVIDUAL_FORWARD


## 2. Raw observation integrity (independent, from CSV)

In [3]:
ig = jload(AUD / "phase34_latency_observation_integrity_v1_4.json")
print("rows:", ig["row_count"], "| arms:", ig["arm_count"],
      "| aliases present:", ig["aliases_physically_present"])
print("per-arm counts:", ig["per_arm_counts"])
print("duplicates:", ig["duplicate_key_count"], "| missing:", ig["missing_count"],
      "| non-finite:", ig["non_finite_latency_count"],
      "| non-positive:", ig["non_positive_latency_count"])
print("global index 0..999 per arm:", ig["global_iteration_index_0_999"])
print("task counts exact 250 each:", ig["task_counts_exact_250_each"])
print("integrity pass:", ig["all_pass"])


rows: 6000 | arms: 6 | aliases present: False
per-arm counts: {'GLOBAL_ONLY_C1': 1000, 'R_PSI_COMPONENT': 1000, 'B_PHI_COMPONENT': 1000, 'S_OMEGA_COMPONENT': 1000, 'SKIP_CONTROLLED_PATH': 1000, 'EXECUTE_CONTROLLED_PATH': 1000}
duplicates: 0 | missing: 0 | non-finite: 0 | non-positive: 0
global index 0..999 per arm: True
task counts exact 250 each: True
integrity pass: True


## 3. Latency summary (six canonical arms)

In [4]:
sm = tshow("phase34_batch1_latency_results_v1_4.csv")


phase34_batch1_latency_results_v1_4.csv | shape: (6, 16)
                    arm    n  batch_size  warmup_count   mean_ms   std_ms    p50_ms    p90_ms    p95_ms    p99_ms    min_ms    max_ms                                                                                            timer_type          precision_semantics                                               task_schedule                   hardware
         GLOBAL_ONLY_C1 1000           1            50 15.767750 0.639550 15.721472 16.699699 16.830668 17.085440 14.539776 18.619392 torch.cuda.Event(enable_timing=True) per individual forward + torch.cuda.synchronize() once per block component-specific canonical global_iteration_index mod 4 (0=ISIC,1=BUSI,2=KVASIR,3=KUB) NVIDIA GeForce RTX 3060 Ti
        R_PSI_COMPONENT 1000           1            50  4.533610 0.730640  4.374016  5.271757  5.702349  7.662858  3.104768  9.807872 torch.cuda.Event(enable_timing=True) per individual forward + torch.cuda.synchronize() once per block com

## 4. Independent summary reproduction

In [5]:
rp = jload(AUD / "phase34_latency_summary_reproduction_v1_4.json")
print("means reproduced   :", rp["means_reproduced"], "/ 6")
print("std reproduced     :", rp["std_reproduced"], "/ 6")
print("p50 reproduced     :", rp["p50_reproduced"], "/ 6")
print("p90 reproduced     :", rp["p90_reproduced"], "/ 6")
print("p95 reproduced     :", rp["p95_reproduced"], "/ 6")
print("p99 reproduced     :", rp["p99_reproduced"], "/ 6")
print("min reproduced     :", rp["min_reproduced"], "/ 6")
print("max reproduced     :", rp["max_reproduced"], "/ 6")
print("all fields 6/6    :", rp["all_fields_6_of_6"])


means reproduced   : 6 / 6
std reproduced     : 6 / 6
p50 reproduced     : 6 / 6
p90 reproduced     : 6 / 6
p95 reproduced     : 6 / 6
p99 reproduced     : 6 / 6
min reproduced     : 6 / 6
max reproduced     : 6 / 6
all fields 6/6    : True


## 5. Per-forward latency distribution (illustrative recomputation from raw CSV)

In [6]:
obs = pd.read_csv(TAB / "phase34_batch1_latency_observations_v1_4.csv")
print("raw observation rows:", len(obs))
for arm in ["GLOBAL_ONLY_C1", "R_PSI_COMPONENT", "B_PHI_COMPONENT", "S_OMEGA_COMPONENT",
            "SKIP_CONTROLLED_PATH", "EXECUTE_CONTROLLED_PATH"]:
    v = obs.loc[obs["arm"] == arm, "latency_ms"].to_numpy(dtype=np.float64)
    q = np.quantile(v, [0.50, 0.90, 0.95, 0.99], method="linear")
    print(f"  {arm:24s} n={v.size} mean={np.mean(v):.6f} std={np.std(v, ddof=1):.6f} "
          f"p50={q[0]:.6f} p90={q[1]:.6f} p95={q[2]:.6f} p99={q[3]:.6f} "
          f"min={np.min(v):.6f} max={np.max(v):.6f}")


raw observation rows: 6000
  GLOBAL_ONLY_C1           n=1000 mean=15.767750 std=0.639550 p50=15.721472 p90=16.699699 p95=16.830668 p99=17.085440 min=14.539776 max=18.619392
  R_PSI_COMPONENT          n=1000 mean=4.533610 std=0.730640 p50=4.374016 p90=5.271757 p95=5.702349 p99=7.662858 min=3.104768 max=9.807872
  B_PHI_COMPONENT          n=1000 mean=0.372810 std=0.081357 p50=0.361472 p90=0.429318 p95=0.501811 p99=0.688210 min=0.037888 max=1.118208
  S_OMEGA_COMPONENT        n=1000 mean=0.254867 std=0.077745 p50=0.243712 p90=0.313344 p95=0.377037 p99=0.516239 min=0.019456 max=1.051648
  SKIP_CONTROLLED_PATH     n=1000 mean=15.718351 std=0.590120 p50=15.684608 p90=16.611431 p95=16.768204 p99=16.916530 min=14.544896 max=17.231873
  EXECUTE_CONTROLLED_PATH  n=1000 mean=18.894091 std=0.658472 p50=18.886640 p90=19.621887 p95=19.833861 p99=20.465858 min=17.303553 max=23.586817


## 6. Conditional weighted runtime

In [7]:
tshow("phase34_conditional_runtime_estimate_v1_4.csv")


phase34_conditional_runtime_estimate_v1_4.csv | shape: (7, 4)
         quantity                                                            value     unit                                                                                                                                                                                                       note
   L_SKIP_mean_ms                                                15.71835146522522       ms                                                                                                                                                                   SKIP = G + B_phi (v1.4 per-forward mean)
L_EXECUTE_mean_ms                                               18.894091419219972       ms                                                                                                                                              EXECUTE = G + B_phi + R_psi + S_omega (v1.4 per-forward mean)
           p_SKIP                                    

,quantity,value,unit,note
0,L_SKIP_mean_ms,15.71835146522522,ms,SKIP = G + B_phi (v1.4 per-forward mean)
1,L_EXECUTE_mean_ms,18.894091419219972,ms,EXECUTE = G + B_phi + R_psi + S_omega (v1.4 pe...
2,p_SKIP,0.7257142857142858,fraction,635/875
3,p_EXECUTE,0.2742857142857143,fraction,240/875 (ETA 236 + ETR 4)
4,L_weighted_ms,16.589411566892352,ms,WEIGHTED_HARDWARE_SPECIFIC_RUNTIME_ESTIMATE; N...
5,formula,L_weighted_v1_4 = (635/875)*mean(SKIP) + (240/...,NaN,MEAN ONLY - no weighted percentiles computed
6,classification,WEIGHTED_HARDWARE_SPECIFIC_RUNTIME_ESTIMATE,NaN,descriptive engineering estimate


## 7. v1.4 vs v1.3 salvage (descriptive only)

In [8]:
tshow("phase34_v1_4_vs_v1_3_salvage_comparison.csv")
print()
print("NOTE: descriptive only - no acceptance range, no rerun trigger, no best-run selection.")
print("v1.3 has no valid per-iteration std/percentiles, so none are compared.")


phase34_v1_4_vs_v1_3_salvage_comparison.csv | shape: (6, 6)
                    arm  v1_4_protocol_compliant_mean_ms  v1_3_salvaged_block_average_mean_ms  absolute_difference_ms  relative_difference                                 note
         GLOBAL_ONLY_C1                        15.767750                            15.602622                0.165129             0.010583 DESCRIPTIVE_ONLY_NO_ACCEPTANCE_RANGE
        R_PSI_COMPONENT                         4.533610                             4.687589               -0.153980            -0.032848 DESCRIPTIVE_ONLY_NO_ACCEPTANCE_RANGE
        B_PHI_COMPONENT                         0.372810                             0.384549               -0.011739            -0.030526 DESCRIPTIVE_ONLY_NO_ACCEPTANCE_RANGE
      S_OMEGA_COMPONENT                         0.254867                             0.225486                0.029381             0.130302 DESCRIPTIVE_ONLY_NO_ACCEPTANCE_RANGE
   SKIP_CONTROLLED_PATH                        15.718351    

## 8. Preserved valid v1.3 non-latency evidence (reused, not regenerated)

In [9]:
tshow("phase34_batch8_throughput_results_v1_3.csv")
tshow("phase34_gpu_memory_results_v1_3.csv")
tshow("phase34_static_resource_profile_v1_3.csv")
tshow("phase34_analytical_compute_summary_v1_3.csv")


phase34_batch8_throughput_results_v1_3.csv | shape: (6, 10)
                    arm  batch_size  measured_batches  processed_samples  synchronized_seconds  samples_per_second                                fps_definition          precision_semantics                timing_scope                                                  hardware
         GLOBAL_ONLY_C1           8              1000               8000             85.414416           93.661005 processed_samples / synchronized_wall_seconds component-specific canonical  MODEL_COMPUTE_PATH_RUNTIME NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,232 B VRAM)
        R_PSI_COMPONENT           8              1000               8000             13.321764          600.521068 processed_samples / synchronized_wall_seconds component-specific canonical SYNTHETIC_COMPONENT_RUNTIME NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,232 B VRAM)
        B_PHI_COMPONENT           8              1000               8000              0.386081        20721.0457

,reference,formula,macs,router_present,refiner_present,s_omega_present,label
0,LITERAL_GLOBAL_ONLY_C1,MAC(G),3.155174e+10,False,False,False,ANALYTICAL_MAC_LOWER_BOUND
1,LITERAL_ALWAYS_REFINE_C2,MAC(G)+MAC(R_psi),3.478240e+10,False,True,False,ANALYTICAL_MAC_LOWER_BOUND
2,FINAL_A8_C9_EXPECTED_CONDITIONAL,MAC(G)+MAC(B_phi)+p_activate*(MAC(R_psi)+MAC(S...,3.243790e+10,True,True,True,ANALYTICAL_MAC_LOWER_BOUND
3,GATED_EXECUTE_ALL_C9,MAC(G)+MAC(B_phi)+MAC(R_psi)+MAC(S_omega),3.478244e+10,True,True,True,ANALYTICAL_MAC_LOWER_BOUND
4,RATIO_A8C9_vs_C1,E/C1,1.028086e+00,NaN,NaN,NaN,ANALYTICAL_RATIO
5,RATIO_A8C9_vs_literal_C2,E/C2,9.325952e-01,NaN,NaN,NaN,ANALYTICAL_RATIO
6,SAVING_vs_literal_C2,C2-E,2.344502e+09,NaN,NaN,NaN,ANALYTICAL_SAVING_MACS
7,RATIO_A8C9_vs_gated_C9,E/gated,9.325942e-01,NaN,NaN,NaN,ANALYTICAL_RATIO
8,SAVING_vs_gated_C9,gated-E,2.344539e+09,NaN,NaN,NaN,ANALYTICAL_SAVING_MACS


## 9. Repaired composite result, audit and final check

In [10]:
res = jload(AUD / "phase34_efficiency_result_v1_4.json")
print("v1.3 latency superseded:", res["v1_3_latency_superseded_due_measurement_granularity_defect"])
print("v1.3 throughput reused valid:", res["v1_3_throughput_reused_valid"])
print("v1.3 memory reused valid:", res["v1_3_memory_reused_valid"])
print("v1.3 static resources reused valid:", res["v1_3_static_resources_reused_valid"])
print("v1.3 analytical MAC reused valid:", res["v1_3_analytical_MAC_reused_valid"])
print("claims not made:", res["claims_not_made"])
print()
au = jload(AUD / "phase34_latency_execution_audit_v1_4.json")
print("audit all_pass:", au["all_pass"], "| throughput reruns:", au["throughput_reruns"],
      "| memory reruns:", au["memory_reruns"])
fc = jload(AUD / "phase34_latency_execution_final_check_v1_4.json")
print("final check:", fc["n_checks"], "checks,", fc["n_failed"], "failed")
term = jload(AUD / "phase34_latency_execution_terminal_v1_4.json")
print()
print("terminal:")
print(" ", term["terminal"])


v1.3 latency superseded: True
v1.3 throughput reused valid: True
v1.3 memory reused valid: True
v1.3 static resources reused valid: True
v1.3 analytical MAC reused valid: True
claims not made: ['real-time capability', 'mobile suitability', 'energy efficiency', 'hardware-independent speed', 'clinical deployment readiness', 'END_TO_END_PIPELINE_RUNTIME']

audit all_pass: True | throughput reruns: 0 | memory reruns: 0
final check: 45 checks, 0 failed

terminal:
  PHASE34_LATENCY_REPAIR_EXECUTION_V1_4_COMPLETE__SIX_CANONICAL_BATCH1_ARMS_PROFILED_WITH_1000_PER_FORWARD_CUDA_EVENT_OBSERVATIONS_EACH__6000_RAW_LATENCY_OBSERVATIONS_PERSISTED_AND_SUMMARY_REPRODUCED__ALIASES_NOT_PHYSICALLY_REEXECUTED__V1_3_THROUGHPUT_MEMORY_STATIC_AND_ANALYTICAL_EVIDENCE_PRESERVED__CONDITIONAL_WEIGHTED_RUNTIME_REPAIRED__NO_PROTECTED_RAW_OR_TEST_ACCESS__PHASE34_CLOSURE_ELIGIBLE_NOT_STARTED


---
## Summary

| item | value |
|---|---|
| physical latency arms | 6 (aliases not physically re-executed) |
| observations | 1000 per arm · 6000 raw rows |
| observation unit | one CUDA-event interval per individual forward |
| statistics | mean · sample std (ddof=1) · linear p50/p90/p95/p99 · min · max |
| raw integrity | 0 duplicates · 0 missing · 0 non-finite · 0 non-positive |
| summary reproduction | 6/6 on all eight statistics |
| L_SKIP / L_EXECUTE / L_weighted | per-forward means from v1.4 protocol |
| v1.3 throughput / memory / static / MAC | preserved valid, reused |
| Phase34 closure | NOT performed · Phase35 NOT started · TEST locked |
